# Drive-backed linked resources CUJ — 2026-08-18

**Branch:** `codex/drive-backed-linked-resources`  
**Draft PR:** https://github.com/runmedev/web/pull/326  
**Drive report:** https://drive.google.com/file/d/1yd85vH_OpNgg_RKMcCGJMpKZ9VZcjxzj/view

## Outcome

**PASS.** The implementation builds, the complete affected automated test matrix passes, and the private-media CUJ passed in the in-app Browser against the local branch. The run used Runme session `quick-island` and the documented `runme-web-test@aisre-gdrive-oai-test.iam.gserviceaccount.com` account through `credentials.google.setServiceAccountFromFilePath(...)` followed by `drive.authorize()`. No private key or access token was printed or stored in notebook content.

The browser run exercised real Drive uploads, authenticated downloads, OPFS-backed blob rendering, playback and seeking, safe fallbacks, permission preservation, deletion semantics, access-denied UI, and cross-principal cache isolation.

## Verification matrix

| Area | Result | Evidence |
| --- | --- | --- |
| Resource schema, Drive URL normalization, safe MIME selection | PASS | `linkedResource.test.ts` |
| Existing Drive/HTTPS attachment and local upload transaction | PASS | `linkedResourceAttachments.test.ts`, `driveResource.test.ts` |
| Resumable upload, auth retry, permission/error mapping | PASS | `drive.test.ts`, `driveResource.test.ts` |
| Principal/version cache isolation, commit marker repair, quota/LRU, memory fallback | PASS | `linkedResourceCache.test.ts` |
| Image/video/audio rendering states and object URL cleanup | PASS | `ResourceCell.test.tsx` plus in-app Browser |
| Resource cells excluded from execution paths | PASS | `notebookData.test.ts`, `runmeConsole.test.ts` |
| Markdown sidecar and notebook diff integration | PASS | `serializeNotebookToMarkdown.test.ts` plus production build |
| File picker, Drive picker, drag/drop, clear-cache UI | PASS | `Actions.test.tsx`, `DriveSyncStatusTab.test.tsx` |
| Upload conflict recovery with explicit Retry insertion | PASS | `Actions.test.tsx`, `linkedResourceAttachments.test.ts` |
| `notebooks.attach` App Console/sandbox helper | PASS | `appJsGlobals.test.ts`, `sandboxJsKernel.test.ts`, `aiAgentInstructions.test.ts` |
| Privacy-safe Phase 4 transfer/cache metrics | PASS | `linkedResourceMetrics.test.ts` plus live cache/download metrics |
| Private WebM playback and seeking | PASS | 2.008 s private blob; playback advanced to 0.62 s; seeking advanced 0.10 s |
| Private animated GIF and Ogg audio | PASS | GIF loaded at 160×90; 2.0065 s audio ready from private blobs |
| PDF and active HTML fallback | PASS | PDF rendered as document link card; HTML created no iframe/object/embed/script |
| Access-denied behavior | PASS | Hidden Drive resources retain status and expose Request access plus Open in Drive |
| Cross-principal cache isolation | PASS | Second authorized service account saw zero video/audio/GIF/blob media and Request access for every private resource |
| Drive permissions and deletion semantics | PASS | All fixture files had zero `anyone` permissions; deleting/relinking a cell retained the Drive file and did not re-upload |

### Automated results

- `pnpm -C app exec vitest run <15 affected suites>`: **15 files, 255 tests passed**.
- `runme run build test`: **passed** after the browser-CUJ fixes (production build and 7/7 repository test-task tests).
- `pnpm exec prettier --check <review files>`: **passed**.
- `git diff --check`: **passed**.
- Repository lint could not be run because the locked workspace does not install an `eslint` binary.

## Review findings and fixes

1. **Cache budget accounting used unrelated origin storage.** The quota-fraction budget was calculated from total browser-origin usage, so unrelated IndexedDB/OPFS data could evict linked media prematurely. Fixed by calculating the linked-media budget from committed linked-resource records while retaining the absolute origin-quota guard. Added regression coverage.
2. **Upload followed by notebook persistence failure could leave a misleading in-memory cell.** Fixed by removing the unsaved cell best-effort and returning a structured conflict error that preserves the uploaded Drive URL for recovery. Added regression coverage.
3. **The new automation method was missing from agent guidance and sandbox coverage.** Added `notebooks.attach` guidance and an end-to-end sandbox bridge test.
4. **The design promised Retry insertion, but the UI only displayed an error string.** Added a durable recovery notice with Open uploaded file and Retry insertion actions. Retry attaches the already-uploaded Drive object and never uploads the bytes again.
5. **Phase 4 metrics were missing.** Added privacy-safe events for upload failures, download latency, cache hits, eviction, and memory fallback. The metric payloads contain no resource URI, filename, token, object URL, or response body.
6. **Drive hides inaccessible files behind 404 responses.** The resource card reported File not found but omitted the design's Request access action. `NOT_FOUND` now offers Request access while preserving the precise status, with component coverage.
7. **Replacing service-account credentials could reuse a token minted for the previous account.** Both service-account file loaders now force token refresh after changing credentials, with regression coverage. WebMCP tests continue to follow the documented `setServiceAccountFromFilePath(...)` plus `drive.authorize()` sequence.

No GitHub review comments were present at the time of this run.

## Browser CUJ result

The private fixture set from the design—WebM, animated GIF, Ogg audio, PDF, and unsupported HTML—was uploaded through `notebooks.attach` to the report's sibling Drive asset folder and rendered in the in-app Browser. Playback, seeking, access-denied behavior, cross-principal cache isolation, absence of public permission creation, active-content link-card fallback, and deletion-retains-file behavior all passed.

The report was returned to the documented Drive test account and the two synthetic negative-test cells were removed, leaving the five intended resource cells. The branch is working end to end for the scoped design and test matrix.